In [ ]:
from pathlib import Path
import pandas as pd

training_folder = Path("../data/PHM2016_CMP/CMP-data/training")
training_files = sorted(training_folder.glob("CMP-training-*.csv"))

print("학습 파일 개수:", len(training_files))
print("첫 번째 파일:", training_files[0])

In [ ]:
first_file = training_files[0]
df = pd.read_csv(first_file)

print("데이터 크기:", df.shape)
display(df.head())

In [ ]:
selected_columns = [
    "TIMESTAMP",
    "WAFER_ID",
    "STAGE",
    "CHAMBER",
    "MAIN_OUTER_AIR_BAG_PRESSURE",
    "CENTER_AIR_BAG_PRESSURE",
    "RETAINER_RING_PRESSURE",
    "SLURRY_FLOW_LINE_A",
    "SLURRY_FLOW_LINE_B",
    "SLURRY_FLOW_LINE_C",
    "WAFER_ROTATION",
    "STAGE_ROTATION",
    "HEAD_ROTATION"
]

selected_df = df[selected_columns]

print("추출 전 데이터 크기:", df.shape)
print("추출 후 데이터 크기:", selected_df.shape)
display(selected_df.head())

In [ ]:
wafer_list = selected_df[["WAFER_ID", "STAGE"]].drop_duplicates()

print("웨이퍼·단계 조합 개수:", len(wafer_list))
display(wafer_list)

In [ ]:
target_path = Path("../data/PHM2016_CMP/CMP-training-removalrate.csv")
target_df = pd.read_csv(target_path)

print("전체 제거율 정답 개수:", len(target_df))
display(target_df.head())

In [ ]:
matched_target = wafer_list.merge(
    target_df,
    on=["WAFER_ID", "STAGE"],
    how="left"
)

print("웨이퍼·단계 조합:", len(wafer_list))
print("연결된 평균 제거율:", len(matched_target))
display(matched_target)

In [ ]:
process_columns = [
    "MAIN_OUTER_AIR_BAG_PRESSURE",
    "CENTER_AIR_BAG_PRESSURE",
    "RETAINER_RING_PRESSURE",
    "SLURRY_FLOW_LINE_A",
    "SLURRY_FLOW_LINE_B",
    "SLURRY_FLOW_LINE_C",
    "WAFER_ROTATION",
    "STAGE_ROTATION",
    "HEAD_ROTATION"
]

# 웨이퍼별 공정변수 평균
mean_features = (
    selected_df
    .groupby(["WAFER_ID", "STAGE"])[process_columns]
    .mean()
    .add_suffix("_mean")
)

# 웨이퍼별 공정변수 표준편차
std_features = (
    selected_df
    .groupby(["WAFER_ID", "STAGE"])[process_columns]
    .std()
    .add_suffix("_std")
)

# 평균과 표준편차를 하나의 표로 결합
wafer_features = pd.concat(
    [mean_features, std_features],
    axis=1
).reset_index()

print("웨이퍼 특징표 크기:", wafer_features.shape)
display(wafer_features)

In [ ]:
model_df = wafer_features.merge(
    target_df,
    on=["WAFER_ID", "STAGE"],
    how="left"
)

print("최종 분석표 크기:", model_df.shape)
print("제거율이 없는 행:", model_df["AVG_REMOVAL_RATE"].isna().sum())

display(model_df)

In [ ]:
all_parts = []

for file in training_files:
    part = pd.read_csv(file, usecols=selected_columns)
    all_parts.append(part)

all_raw_df = pd.concat(all_parts, ignore_index=True)

print("읽은 파일 개수:", len(all_parts))
print("전체 데이터 크기:", all_raw_df.shape)
display(all_raw_df.head())

In [ ]:
all_mean_features = (
    all_raw_df
    .groupby(["WAFER_ID", "STAGE"])[process_columns]
    .mean()
    .add_suffix("_mean")
)

all_std_features = (
    all_raw_df
    .groupby(["WAFER_ID", "STAGE"])[process_columns]
    .std()
    .add_suffix("_std")
)

all_wafer_features = pd.concat(
    [all_mean_features, all_std_features],
    axis=1
).reset_index()

print("전체 웨이퍼 특징표 크기:", all_wafer_features.shape)
display(all_wafer_features.head())

In [ ]:
full_model_df = all_wafer_features.merge(
    target_df,
    on=["WAFER_ID", "STAGE"],
    how="left"
)

print("최종 분석 데이터 크기:", full_model_df.shape)
print("제거율이 없는 행:", full_model_df["AVG_REMOVAL_RATE"].isna().sum())

display(full_model_df.head())

In [ ]:
base_name_ko = {
    "MAIN_OUTER_AIR_BAG_PRESSURE": "외곽_에어백_압력",
    "CENTER_AIR_BAG_PRESSURE": "중앙_에어백_압력",
    "RETAINER_RING_PRESSURE": "리테이너링_압력",
    "SLURRY_FLOW_LINE_A": "슬러리_A_유량",
    "SLURRY_FLOW_LINE_B": "슬러리_B_유량",
    "SLURRY_FLOW_LINE_C": "슬러리_C_유량",
    "WAFER_ROTATION": "웨이퍼_회전값",
    "STAGE_ROTATION": "스테이지_회전값",
    "HEAD_ROTATION": "헤드_회전값"
}

feature_name_ko = {
    "WAFER_ID": "웨이퍼_ID",
    "STAGE": "공정단계"
}

for english_name, korean_name in base_name_ko.items():
    feature_name_ko[f"{english_name}_mean"] = f"{korean_name}_평균"
    feature_name_ko[f"{english_name}_std"] = f"{korean_name}_표준편차"

all_wafer_features_ko = all_wafer_features.rename(
    columns=feature_name_ko
)

display(all_wafer_features_ko.head())

In [ ]:
model_name_ko = feature_name_ko.copy()
model_name_ko["AVG_REMOVAL_RATE"] = "평균_제거율"

full_model_df_ko = full_model_df.rename(
    columns=model_name_ko
)

display(full_model_df_ko.head())

In [ ]:
full_model_df = all_wafer_features.merge(
    target_df,
    on=["WAFER_ID", "STAGE"],
    how="left"
)

print("최종 분석 데이터 크기:", full_model_df.shape)
print("제거율이 없는 행:", full_model_df["AVG_REMOVAL_RATE"].isna().sum())

display(full_model_df.head())

In [ ]:
print("전체 행·열:", full_model_df.shape)
print("중복 웨이퍼·단계:", full_model_df.duplicated(["WAFER_ID", "STAGE"]).sum())
print("전체 결측치:", full_model_df.isna().sum().sum())
print("제거율 최솟값:", full_model_df["AVG_REMOVAL_RATE"].min())
print("제거율 최댓값:", full_model_df["AVG_REMOVAL_RATE"].max())

In [ ]:
output_path = Path("../results/full_model_data.csv")

full_model_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", output_path)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))

sns.histplot(
    data=full_model_df,
    x="AVG_REMOVAL_RATE",
    bins=30,
    kde=True
)

plt.title("Distribution of Average Removal Rate")
plt.xlabel("Average Removal Rate")
plt.ylabel("Number of Wafers")
plt.show()

In [ ]:
stage_summary = (
    full_model_df
    .groupby("STAGE")["AVG_REMOVAL_RATE"]
    .agg(["count", "mean", "std", "min", "max"])
)

display(stage_summary)

In [ ]:
plt.figure(figsize=(7, 5))

sns.boxplot(
    data=full_model_df,
    x="STAGE",
    y="AVG_REMOVAL_RATE"
)

plt.title("Removal Rate by CMP Stage")
plt.xlabel("CMP Stage")
plt.ylabel("Average Removal Rate")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.scatterplot(
    data=full_model_df,
    x="MAIN_OUTER_AIR_BAG_PRESSURE_mean",
    y="AVG_REMOVAL_RATE",
    hue="STAGE",
    ax=axes[0]
)
axes[0].set_title("Pressure vs Removal Rate")
axes[0].set_xlabel("Average Outer Air Bag Pressure")
axes[0].set_ylabel("Average Removal Rate")

sns.scatterplot(
    data=full_model_df,
    x="SLURRY_FLOW_LINE_C_mean",
    y="AVG_REMOVAL_RATE",
    hue="STAGE",
    ax=axes[1]
)
axes[1].set_title("Slurry Flow vs Removal Rate")
axes[1].set_xlabel("Average Slurry C Flow")
axes[1].set_ylabel("Average Removal Rate")

sns.scatterplot(
    data=full_model_df,
    x="HEAD_ROTATION_mean",
    y="AVG_REMOVAL_RATE",
    hue="STAGE",
    ax=axes[2]
)
axes[2].set_title("Head Rotation vs Removal Rate")
axes[2].set_xlabel("Average Head Rotation")
axes[2].set_ylabel("Average Removal Rate")

plt.tight_layout()
plt.show()

In [ ]:
check_columns = [
    "MAIN_OUTER_AIR_BAG_PRESSURE_mean",
    "SLURRY_FLOW_LINE_C_mean",
    "HEAD_ROTATION_mean",
    "AVG_REMOVAL_RATE"
]

correlation = full_model_df[check_columns].corr()

display(correlation.round(3))

In [ ]:
plt.figure(figsize=(8, 6))

sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    fmt=".2f"
)

plt.title("Correlation Between Process Variables and Removal Rate")
plt.show()

In [ ]:
removal_correlation = (
    correlation["AVG_REMOVAL_RATE"]
    .drop("AVG_REMOVAL_RATE")
    .sort_values(key=abs, ascending=False)
)

display(removal_correlation)

In [ ]:
wafer_chamber = (
    all_raw_df
    .groupby(["WAFER_ID", "STAGE"])["CHAMBER"]
    .first()
    .reset_index()
)

analysis_df = full_model_df.merge(
    wafer_chamber,
    on=["WAFER_ID", "STAGE"],
    how="left"
)

group_summary = (
    analysis_df
    .groupby(["STAGE", "CHAMBER"])["AVG_REMOVAL_RATE"]
    .agg(["count", "mean", "std", "min", "max"])
    .round(2)
)

display(group_summary)

In [ ]:
outlier_check = analysis_df[
    (analysis_df["STAGE"] == "A") &
    (analysis_df["CHAMBER"] == 1.0)
].sort_values(
    "AVG_REMOVAL_RATE",
    ascending=False
)

display(
    outlier_check[
        ["WAFER_ID", "STAGE", "CHAMBER", "AVG_REMOVAL_RATE"]
    ].head(10)
)

In [ ]:
robust_summary = (
    analysis_df
    .groupby(["STAGE", "CHAMBER"])["AVG_REMOVAL_RATE"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(2)
)

display(robust_summary)

In [ ]:
# Stage·Chamber별 제거율의 Q1과 Q3 계산
q1 = analysis_df.groupby(
    ["STAGE", "CHAMBER"]
)["AVG_REMOVAL_RATE"].transform(lambda x: x.quantile(0.25))

q3 = analysis_df.groupby(
    ["STAGE", "CHAMBER"]
)["AVG_REMOVAL_RATE"].transform(lambda x: x.quantile(0.75))

# IQR과 이상치 판정 범위 계산
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

# 원본을 보존하고 이상치 여부 열 추가
analysis_flagged_df = analysis_df.copy()

analysis_flagged_df["OUTLIER"] = (
    (analysis_flagged_df["AVG_REMOVAL_RATE"] < lower_bound) |
    (analysis_flagged_df["AVG_REMOVAL_RATE"] > upper_bound)
)

print(
    "이상치 후보 개수:",
    analysis_flagged_df["OUTLIER"].sum()
)

display(
    analysis_flagged_df[
        analysis_flagged_df["OUTLIER"]
    ][
        ["WAFER_ID", "STAGE", "CHAMBER", "AVG_REMOVAL_RATE"]
    ].sort_values(
        "AVG_REMOVAL_RATE",
        ascending=False
    )
)

In [ ]:
analysis_clean_df = analysis_flagged_df[
    analysis_flagged_df["OUTLIER"] == False
].copy()

print("제외 전 웨이퍼 수:", len(analysis_flagged_df))
print("이상치 후보 수:", analysis_flagged_df["OUTLIER"].sum())
print("제외 후 웨이퍼 수:", len(analysis_clean_df))

In [ ]:
before_summary = (
    analysis_flagged_df
    .groupby(["STAGE", "CHAMBER"])["AVG_REMOVAL_RATE"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(2)
)

after_summary = (
    analysis_clean_df
    .groupby(["STAGE", "CHAMBER"])["AVG_REMOVAL_RATE"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(2)
)

print("이상치 제외 전")
display(before_summary)

print("이상치 제외 후")
display(after_summary)

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=analysis_clean_df,
    x="STAGE",
    y="AVG_REMOVAL_RATE",
    hue="CHAMBER"
)

plt.title("Removal Rate after Outlier Filtering")
plt.xlabel("CMP Stage")
plt.ylabel("Average Removal Rate")
plt.show()

In [ ]:
variables = [
    "MAIN_OUTER_AIR_BAG_PRESSURE_mean",
    "SLURRY_FLOW_LINE_C_mean",
    "HEAD_ROTATION_mean"
]

group_correlation_list = []

for (stage, chamber), group in analysis_clean_df.groupby(
    ["STAGE", "CHAMBER"]
):
    correlation_values = (
        group[variables + ["AVG_REMOVAL_RATE"]]
        .corr()["AVG_REMOVAL_RATE"]
        .drop("AVG_REMOVAL_RATE")
    )

    for variable, correlation_value in correlation_values.items():
        group_correlation_list.append({
            "STAGE": stage,
            "CHAMBER": chamber,
            "VARIABLE": variable,
            "CORRELATION": correlation_value
        })

group_correlation_df = pd.DataFrame(group_correlation_list)

display(
    group_correlation_df.pivot(
        index="VARIABLE",
        columns=["STAGE", "CHAMBER"],
        values="CORRELATION"
    ).round(3)
)

In [ ]:
wear_columns = [
    "USAGE_OF_BACKING_FILM",
    "USAGE_OF_DRESSER",
    "USAGE_OF_POLISHING_TABLE",
    "USAGE_OF_DRESSER_TABLE",
    "USAGE_OF_MEMBRANE",
    "USAGE_OF_PRESSURIZED_SHEET"
]

all_columns = [
    "WAFER_ID",
    "STAGE",
    "CHAMBER"
] + process_columns + wear_columns

In [ ]:
expanded_parts = []

for file in training_files:
    part = pd.read_csv(file, usecols=all_columns)
    expanded_parts.append(part)

expanded_raw_df = pd.concat(
    expanded_parts,
    ignore_index=True
)

print("확장 원천 데이터 크기:", expanded_raw_df.shape)
display(expanded_raw_df.head())

In [ ]:
# 압력·유량·회전: 평균값
expanded_process_mean = (
    expanded_raw_df
    .groupby(["WAFER_ID", "STAGE"])[process_columns]
    .mean()
    .add_suffix("_mean")
)

# 압력·유량·회전: 공정 중 변동성
expanded_process_std = (
    expanded_raw_df
    .groupby(["WAFER_ID", "STAGE"])[process_columns]
    .std()
    .add_suffix("_std")
)

# 패드·드레서 등 누적 사용상태
wear_features = (
    expanded_raw_df
    .groupby(["WAFER_ID", "STAGE"])[wear_columns]
    .mean()
    .add_suffix("_mean")
)

# 해당 웨이퍼가 사용한 챔버
chamber_feature = (
    expanded_raw_df
    .groupby(["WAFER_ID", "STAGE"])["CHAMBER"]
    .first()
)

# 모든 특징을 하나의 표로 결합
expanded_wafer_features = pd.concat(
    [
        chamber_feature,
        expanded_process_mean,
        expanded_process_std,
        wear_features
    ],
    axis=1
).reset_index()

print("확장 웨이퍼 특징표 크기:", expanded_wafer_features.shape)
display(expanded_wafer_features.head())

In [ ]:
expanded_model_df = expanded_wafer_features.merge(
    target_df,
    on=["WAFER_ID", "STAGE"],
    how="left"
)

print("최종 확장 데이터 크기:", expanded_model_df.shape)
print(
    "제거율이 없는 행:",
    expanded_model_df["AVG_REMOVAL_RATE"].isna().sum()
)

display(expanded_model_df.head())

In [ ]:
expanded_output_path = Path(
    "../results/expanded_model_data.csv"
)

expanded_model_df.to_csv(
    expanded_output_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", expanded_output_path)

In [ ]:
all_mean_features = (
    all_raw_df
    .groupby(["WAFER_ID", "STAGE"])[process_columns]
    .mean()
    .add_suffix("_mean")
)

all_std_features = (
    all_raw_df
    .groupby(["WAFER_ID", "STAGE"])[process_columns]
    .std()
    .add_suffix("_std")
)

all_wafer_features = pd.concat(
    [all_mean_features, all_std_features],
    axis=1
).reset_index()

print("전체 웨이퍼 특징표 크기:", all_wafer_features.shape)
display(all_wafer_features.head())

In [ ]:
# 앞에서 이상치 제외 후 남긴 웨이퍼 목록
clean_keys = analysis_clean_df[
    ["WAFER_ID", "STAGE"]
].drop_duplicates()

# 확장 데이터에서 정상 범위 웨이퍼만 선택
model_ready_df = expanded_model_df.merge(
    clean_keys,
    on=["WAFER_ID", "STAGE"],
    how="inner"
)

print("모델링 데이터 크기:", model_ready_df.shape)
print("전체 결측치:", model_ready_df.isna().sum().sum())

In [ ]:
X = model_ready_df.drop(
    columns=["WAFER_ID", "AVG_REMOVAL_RATE"]
)

y = model_ready_df["AVG_REMOVAL_RATE"]

print("입력 데이터:", X.shape)
print("정답 데이터:", y.shape)
display(X.head())

In [ ]:
from sklearn.model_selection import train_test_split

group_label = (
    X["STAGE"].astype(str)
    + "_"
    + X["CHAMBER"].astype(str)
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=group_label
)

print("학습용 웨이퍼:", X_train.shape)
print("시험용 웨이퍼:", X_test.shape)

In [ ]:
from sklearn.model_selection import train_test_split

group_label = (
    X["STAGE"].astype(str)
    + "_"
    + X["CHAMBER"].astype(str)
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=group_label
)

print("학습용 웨이퍼:", X_train.shape)
print("시험용 웨이퍼:", X_test.shape)

In [ ]:
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.3f}")
print(f"RMSE : {rmse:.3f}")
print(f"R²   : {r2:.3f}")

In [ ]:
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.3f}")
print(f"RMSE : {rmse:.3f}")
print(f"R²   : {r2:.3f}")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

# 문자로 된 분류 변수
categorical_columns = ["STAGE", "CHAMBER"]

# 나머지는 숫자형 공정변수
numeric_columns = [
    column for column in X.columns
    if column not in categorical_columns
]

# 숫자와 문자 데이터를 모델이 학습할 형태로 변환
preprocessor = ColumnTransformer([
    (
        "numeric",
        SimpleImputer(strategy="median"),
        numeric_columns
    ),
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_columns
    )
])

# 전처리와 Random Forest를 하나로 연결
model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "regressor",
        RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        )
    )
])

# 학습용 80% 데이터로 관계 학습
model.fit(X_train, y_train)

print("Random Forest 모델 학습 완료")

In [ ]:
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.3f}")
print(f"RMSE : {rmse:.3f}")
print(f"R²   : {r2:.3f}")

In [ ]:
# 학습 데이터의 Stage·Chamber별 평균 제거율 계산
baseline_train = X_train[["STAGE", "CHAMBER"]].copy()
baseline_train["AVG_REMOVAL_RATE"] = y_train

group_mean = baseline_train.groupby(
    ["STAGE", "CHAMBER"]
)["AVG_REMOVAL_RATE"].mean()

# 시험 데이터에는 해당 그룹의 평균 제거율만 예측
baseline_pred = [
    group_mean.loc[(stage, chamber)]
    for stage, chamber in zip(
        X_test["STAGE"],
        X_test["CHAMBER"]
    )
]

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(
    mean_squared_error(y_test, baseline_pred)
)
baseline_r2 = r2_score(y_test, baseline_pred)

print("Stage·Chamber 평균 기준모델")
print(f"MAE  : {baseline_mae:.3f}")
print(f"RMSE : {baseline_rmse:.3f}")
print(f"R²   : {baseline_r2:.3f}")

print("\nRandom Forest")
print(f"MAE  : {mae:.3f}")
print(f"RMSE : {rmse:.3f}")
print(f"R²   : {r2:.3f}")

In [ ]:
comparison_df = pd.DataFrame({
    "실제_제거율": y_test.values,
    "예측_제거율": y_pred
})

plt.figure(figsize=(7, 6))

sns.scatterplot(
    data=comparison_df,
    x="실제_제거율",
    y="예측_제거율"
)

minimum = min(
    comparison_df["실제_제거율"].min(),
    comparison_df["예측_제거율"].min()
)

maximum = max(
    comparison_df["실제_제거율"].max(),
    comparison_df["예측_제거율"].max()
)

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    color="red",
    linestyle="--",
    label="Perfect prediction"
)

plt.title("Actual vs Predicted Removal Rate")
plt.xlabel("Actual Removal Rate")
plt.ylabel("Predicted Removal Rate")
plt.legend()
plt.show()

In [ ]:
error_df = model_ready_df.loc[
    X_test.index,
    ["WAFER_ID", "STAGE", "CHAMBER", "AVG_REMOVAL_RATE"]
].copy()

error_df["PREDICTED_REMOVAL_RATE"] = y_pred
error_df["ERROR"] = (
    error_df["PREDICTED_REMOVAL_RATE"]
    - error_df["AVG_REMOVAL_RATE"]
)
error_df["ABS_ERROR"] = error_df["ERROR"].abs()

display(
    error_df.sort_values(
        "ABS_ERROR",
        ascending=False
    ).head(10)
)

In [ ]:
from sklearn.inspection import permutation_importance

importance_result = permutation_importance(
    model,
    X_test,
    y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "변수": X_test.columns,
    "중요도": importance_result.importances_mean
}).sort_values(
    "중요도",
    ascending=False
)

display(importance_df.head(15))

In [ ]:
top_importance = importance_df.head(15)

plt.figure(figsize=(10, 7))

sns.barplot(
    data=top_importance,
    x="중요도",
    y="변수"
)

plt.title("Top 15 Feature Importance")
plt.xlabel("Permutation Importance")
plt.ylabel("Process Variable")
plt.tight_layout()
plt.show()

In [ ]:
# Chamber와 Stage를 제외한 공정·소모품 변수
X_process_only = model_ready_df.drop(
    columns=[
        "WAFER_ID",
        "AVG_REMOVAL_RATE",
        "STAGE",
        "CHAMBER"
    ]
)

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_process_only,
    y,
    test_size=0.2,
    random_state=42
)

process_only_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "regressor",
        RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        )
    )
])

process_only_model.fit(X_train_p, y_train_p)
y_pred_p = process_only_model.predict(X_test_p)

mae_p = mean_absolute_error(y_test_p, y_pred_p)
rmse_p = np.sqrt(mean_squared_error(y_test_p, y_pred_p))
r2_p = r2_score(y_test_p, y_pred_p)

print("Chamber·Stage 제외 모델")
print(f"MAE  : {mae_p:.3f}")
print(f"RMSE : {rmse_p:.3f}")
print(f"R²   : {r2_p:.3f}")

In [ ]:
# 기존 전체 모델과 동일한 학습·시험 웨이퍼 사용
X_train_p = X_process_only.loc[X_train.index]
X_test_p = X_process_only.loc[X_test.index]

y_train_p = y.loc[X_train.index]
y_test_p = y.loc[X_test.index]

process_only_model.fit(X_train_p, y_train_p)
y_pred_p = process_only_model.predict(X_test_p)

mae_p = mean_absolute_error(y_test_p, y_pred_p)
rmse_p = np.sqrt(mean_squared_error(y_test_p, y_pred_p))
r2_p = r2_score(y_test_p, y_pred_p)

print("동일 시험 데이터 기준 비교")
print(f"전체 모델       : MAE={mae:.3f}, RMSE={rmse:.3f}, R²={r2:.3f}")
print(f"공정변수만 모델 : MAE={mae_p:.3f}, RMSE={rmse_p:.3f}, R²={r2_p:.3f}")

In [ ]:
feature_columns = [
    column for column in model_ready_df.columns
    if column not in [
        "WAFER_ID",
        "STAGE",
        "CHAMBER",
        "AVG_REMOVAL_RATE"
    ]
]

group_results = []
group_models = {}

for (stage, chamber), group in model_ready_df.groupby(
    ["STAGE", "CHAMBER"]
):
    X_group = group[feature_columns]
    y_group = group["AVG_REMOVAL_RATE"]

    X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(
        X_group,
        y_group,
        test_size=0.2,
        random_state=42
    )

    group_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                n_jobs=-1
            )
        )
    ])

    group_model.fit(X_train_g, y_train_g)
    y_pred_g = group_model.predict(X_test_g)

    # 비교용: 해당 그룹의 학습 데이터 평균만 예측
    baseline_pred_g = np.full(
        len(y_test_g),
        y_train_g.mean()
    )

    group_results.append({
        "STAGE": stage,
        "CHAMBER": chamber,
        "시험_웨이퍼": len(y_test_g),
        "기준_MAE": mean_absolute_error(
            y_test_g, baseline_pred_g
        ),
        "RF_MAE": mean_absolute_error(
            y_test_g, y_pred_g
        ),
        "RF_RMSE": np.sqrt(
            mean_squared_error(y_test_g, y_pred_g)
        ),
        "RF_R2": r2_score(y_test_g, y_pred_g)
    })

    group_models[(stage, chamber)] = group_model

group_results_df = pd.DataFrame(group_results)
display(group_results_df.round(3))

In [ ]:
validation_folder = Path(
    "../data/PHM2016_CMP_VALIDATION/validation"
)

validation_target_path = Path(
    "../data/PHM2016_CMP_VALIDATION/CMP-validation-removalrate.csv"
)

validation_files = sorted(
    validation_folder.glob("CMP-validation-*.csv")
)

print("Validation Raw 파일:", len(validation_files))
print("제거율 정답파일 존재:", validation_target_path.exists())
print("첫 번째 파일:", validation_files[0])

In [ ]:
validation_parts = []

for file in validation_files:
    part = pd.read_csv(file, usecols=all_columns)

    if not part.empty:
        validation_parts.append(part)

validation_raw_df = pd.concat(
    validation_parts,
    ignore_index=True
)

print("내용이 있는 파일 개수:", len(validation_parts))
print("검증 원천 데이터 크기:", validation_raw_df.shape)

display(validation_raw_df.head())

In [ ]:
# 1. 공정변수: 웨이퍼별 평균
validation_process_mean = (
    validation_raw_df
    .groupby(["WAFER_ID", "STAGE"])[process_columns]
    .mean()
    .add_suffix("_mean")
)

# 2. 공정변수: 웨이퍼별 표준편차
validation_process_std = (
    validation_raw_df
    .groupby(["WAFER_ID", "STAGE"])[process_columns]
    .std()
    .add_suffix("_std")
)

# 3. 소모품 사용량: 웨이퍼별 평균
validation_wear_mean = (
    validation_raw_df
    .groupby(["WAFER_ID", "STAGE"])[wear_columns]
    .mean()
    .add_suffix("_mean")
)

# 4. 웨이퍼가 사용한 챔버 번호
validation_chamber = (
    validation_raw_df
    .groupby(["WAFER_ID", "STAGE"])["CHAMBER"]
    .first()
)

# 5. 네 개의 표를 하나로 결합
validation_features = pd.concat(
    [
        validation_chamber,
        validation_process_mean,
        validation_process_std,
        validation_wear_mean
    ],
    axis=1
).reset_index()

print("검증 웨이퍼 특징표 크기:", validation_features.shape)
display(validation_features.head())

In [ ]:
from sklearn.base import clone

# 1. 기존 학습 데이터 전체를 입력값(X)과 정답(y)으로 분리
X_final = model_ready_df.drop(
    columns=["WAFER_ID", "AVG_REMOVAL_RATE"]
)
y_final = model_ready_df["AVG_REMOVAL_RATE"]

# 2. 앞에서 만든 Random Forest 구조를 복사
final_model = clone(model)

# 3. 기존 학습 데이터 전체로 최종 학습
final_model.fit(X_final, y_final)

# 4. 검증 데이터 열을 학습 데이터와 같은 순서로 맞춤
X_validation = validation_features.reindex(
    columns=X_final.columns
)

print("최종 학습 데이터:", X_final.shape)
print("검증 입력 데이터:", X_validation.shape)
print("누락된 입력값:", X_validation.isna().sum().sum())

# 5. 검증 웨이퍼의 제거율 예측
validation_predictions = final_model.predict(X_validation)

# 6. 예측 결과표 생성
validation_prediction_df = validation_features[
    ["WAFER_ID", "STAGE", "CHAMBER"]
].copy()

validation_prediction_df["PREDICTED_REMOVAL_RATE"] = (
    validation_predictions
)

print("예측한 웨이퍼 개수:", len(validation_prediction_df))
display(validation_prediction_df.head(10))

In [ ]:
from sklearn.base import clone

# 1. 기존 학습 데이터 전체를 입력값(X)과 정답(y)으로 분리
X_final = model_ready_df.drop(
    columns=["WAFER_ID", "AVG_REMOVAL_RATE"]
)
y_final = model_ready_df["AVG_REMOVAL_RATE"]

# 2. 앞에서 만든 Random Forest 구조를 복사
final_model = clone(model)

# 3. 기존 학습 데이터 전체로 최종 학습
final_model.fit(X_final, y_final)

# 4. 검증 데이터 열을 학습 데이터와 같은 순서로 맞춤
X_validation = validation_features.reindex(
    columns=X_final.columns
)

print("최종 학습 데이터:", X_final.shape)
print("검증 입력 데이터:", X_validation.shape)
print("누락된 입력값:", X_validation.isna().sum().sum())

# 5. 검증 웨이퍼의 제거율 예측
validation_predictions = final_model.predict(X_validation)

# 6. 예측 결과표 생성
validation_prediction_df = validation_features[
    ["WAFER_ID", "STAGE", "CHAMBER"]
].copy()

validation_prediction_df["PREDICTED_REMOVAL_RATE"] = (
    validation_predictions
)

print("예측한 웨이퍼 개수:", len(validation_prediction_df))
display(validation_prediction_df.head(10))

In [ ]:
from sklearn.base import clone
import pandas as pd

# 1. 학습 데이터 준비
X_final = model_ready_df.drop(
    columns=["WAFER_ID", "AVG_REMOVAL_RATE"]
).copy()

y_final = model_ready_df["AVG_REMOVAL_RATE"].copy()

# 2. 검증 데이터의 열과 순서를 학습 데이터에 맞춤
X_validation = validation_features.reindex(
    columns=X_final.columns
).copy()

# 3. STAGE는 문자형으로 통일
X_final["STAGE"] = X_final["STAGE"].astype(str).astype(object)
X_validation["STAGE"] = X_validation["STAGE"].astype(str).astype(object)

# 4. STAGE 이외의 모든 열은 숫자형으로 통일
numeric_columns = [
    column for column in X_final.columns
    if column != "STAGE"
]

for column in numeric_columns:
    X_final[column] = pd.to_numeric(
        X_final[column],
        errors="coerce"
    ).astype("float64")

    X_validation[column] = pd.to_numeric(
        X_validation[column],
        errors="coerce"
    ).astype("float64")

# 5. 최종 모델 학습
final_model = clone(model)
final_model.fit(X_final, y_final)

# 6. 데이터 상태 확인
print("최종 학습 데이터:", X_final.shape)
print("검증 입력 데이터:", X_validation.shape)
print("검증 데이터 결측값:", X_validation.isna().sum().sum())

# 7. 제거율 예측
validation_predictions = final_model.predict(X_validation)

# 8. 예측 결과표 생성
validation_prediction_df = validation_features[

In [ ]:
from sklearn.base import clone
import pandas as pd

# 학습 데이터와 검증 데이터 준비
X_final = model_ready_df.drop(
    columns=["WAFER_ID", "AVG_REMOVAL_RATE"]
).copy()

y_final = pd.to_numeric(
    model_ready_df["AVG_REMOVAL_RATE"],
    errors="coerce"
)

X_validation = validation_features.reindex(
    columns=X_final.columns
).copy()

# STAGE는 문자, 나머지 변수는 숫자로 통일
X_final["STAGE"] = X_final["STAGE"].astype(str)
X_validation["STAGE"] = X_validation["STAGE"].astype(str)

for column in X_final.columns:
    if column != "STAGE":
        X_final[column] = pd.to_numeric(
            X_final[column], errors="coerce"
        ).astype("float64")

        X_validation[column] = pd.to_numeric(
            X_validation[column], errors="coerce"
        ).astype("float64")

# 전체 학습 데이터로 최종 모델 학습
final_model = clone(model)
final_model.fit(X_final, y_final)

# 검증 데이터 제거율 예측
validation_predictions = final_model.predict(X_validation)

# 열 3개를 선택해 예측 결과표 생성
validation_prediction_df = validation_features.loc[
    :, ["WAFER_ID", "STAGE", "CHAMBER"]
].copy()

validation_prediction_df["PREDICTED_REMOVAL_RATE"] = validation_predictions

print("학습 데이터:", X_final.shape)
print("검증 데이터:", X_validation.shape)
print("검증 결측값:", X_validation.isna().sum().sum())
print("

In [ ]:
from sklearn.base import clone
import pandas as pd

# 학습 데이터와 검증 데이터 준비
X_final = model_ready_df.drop(
    columns=["WAFER_ID", "AVG_REMOVAL_RATE"]
).copy()

y_final = pd.to_numeric(
    model_ready_df["AVG_REMOVAL_RATE"],
    errors="coerce"
)

X_validation = validation_features.reindex(
    columns=X_final.columns
).copy()

# STAGE는 문자, 나머지 변수는 숫자로 통일
X_final["STAGE"] = X_final["STAGE"].astype(str)
X_validation["STAGE"] = X_validation["STAGE"].astype(str)

for column in X_final.columns:
    if column != "STAGE":
        X_final[column] = pd.to_numeric(
            X_final[column], errors="coerce"
        ).astype("float64")

        X_validation[column] = pd.to_numeric(
            X_validation[column], errors="coerce"
        ).astype("float64")

# 전체 학습 데이터로 최종 모델 학습
final_model = clone(model)
final_model.fit(X_final, y_final)

# 검증 데이터 제거율 예측
validation_predictions = final_model.predict(X_validation)

# 열 3개를 선택해 예측 결과표 생성
validation_prediction_df = validation_features.loc[
    :, ["WAFER_ID", "STAGE", "CHAMBER"]
].copy()

validation_prediction_df["PREDICTED_REMOVAL_RATE"] = validation_predictions

print("학습 데이터:", X_final.shape)
print("검증 데이터:", X_validation.shape)
print("검증 결측값:", X_validation.isna().sum().sum())
print("예측한 웨이퍼:", len(validation_prediction_df))

display(validation_prediction_df.head(10))

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 1. 검증 데이터의 실제 제거율 정답 불러오기
validation_target_df = pd.read_csv(
    validation_target_path
)

# 2. 자료형 통일
validation_target_df["WAFER_ID"] = pd.to_numeric(
    validation_target_df["WAFER_ID"],
    errors="coerce"
)

validation_target_df["STAGE"] = (
    validation_target_df["STAGE"].astype(str)
)

validation_target_df["AVG_REMOVAL_RATE"] = pd.to_numeric(
    validation_target_df["AVG_REMOVAL_RATE"],
    errors="coerce"
)

validation_prediction_df["WAFER_ID"] = pd.to_numeric(
    validation_prediction_df["WAFER_ID"],
    errors="coerce"
)

validation_prediction_df["STAGE"] = (
    validation_prediction_df["STAGE"].astype(str)
)

# 3. 예측값과 실제 제거율

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 1. 검증 데이터의 실제 제거율 정답 불러오기
validation_target_df = pd.read_csv(
    validation_target_path
)

# 2. 자료형 통일
validation_target_df["WAFER_ID"] = pd.to_numeric(
    validation_target_df["WAFER_ID"],
    errors="coerce"
)

validation_target_df["STAGE"] = (
    validation_target_df["STAGE"].astype(str)
)

validation_target_df["AVG_REMOVAL_RATE"] = pd.to_numeric(
    validation_target_df["AVG_REMOVAL_RATE"],
    errors="coerce"
)

validation_prediction_df["WAFER_ID"] = pd.to_numeric(
    validation_prediction_df["WAFER_ID"],
    errors="coerce"
)

validation_prediction_df["STAGE"] = (
    validation_prediction_df["STAGE"].astype(str)
)

# 3. 예측값과 실제 제거율 연결
validation_result_df = validation_prediction_df.merge(
    validation_target_df,
    on=["WAFER_ID", "STAGE"],
    how="inner"
)

# 4. 오차 계산
validation_result_df["ERROR"] = (
    validation_result_df["PREDICTED_REMOVAL_RATE"]
    - validation_result_df["AVG_REMOVAL_RATE"]
)

validation_result_df["ABS_ERROR"] = (
    validation_result_df["ERROR"].abs()
)

# 5. 외부 검증 성능 계산
y_actual = validation_result_df["AVG_REMOVAL_RATE"]
y_predicted = validation_result_df["PREDICTED_REMOVAL_RATE"]

validation_mae = mean_absolute_error(
    y_actual,
    y_predicted
)

validation_rmse = np.sqrt(
    mean_squared_error(y_actual, y_predicted)
)

validation_r2 = r2_score(
    y_actual,
    y_predicted
)

print("검증 정답 개수:", len(validation_target_df))
print("예측값과 연결된 개수:", len(validation_result_df))
print()
print("외부 검증 결과")
print(f"MAE  : {validation_mae:.3f}")
print(f"RMSE : {validation_rmse:.3f}")
print(f"R²   : {validation_r2:.3f}")

display(validation_result_df.head(10))

In [ ]:
# 학습 데이터의 웨이퍼·단계 조합
training_keys = model_ready_df[
    ["WAFER_ID", "STAGE"]
].drop_duplicates()

# 검증 데이터의 웨이퍼·단계 조합
validation_keys = validation_features[
    ["WAFER_ID", "STAGE"]
].drop_duplicates()

# 학습과 검증 양쪽에 모두 존재하는 조합 확인
overlap_keys = training_keys.merge(
    validation_keys,
    on=["WAFER_ID", "STAGE"],
    how="inner"
)

print("학습 웨이퍼·단계 조합:", len(training_keys))
print("검증 웨이퍼·단계 조합:", len(validation_keys))
print("중복된 웨이퍼·단계 조합:", len(overlap_keys))

display(overlap_keys.head(10))

In [ ]:
# 학습 데이터의 웨이퍼·단계 조합
training_keys = model_ready_df[
    ["WAFER_ID", "STAGE"]
].drop_duplicates()

# 검증 데이터의 웨이퍼·단계 조합
validation_keys = validation_features[
    ["WAFER_ID", "STAGE"]
].drop_duplicates()

# 학습과 검증 양쪽에 모두 존재하는 조합 확인
overlap_keys = training_keys.merge(
    validation_keys,
    on=["WAFER_ID", "STAGE"],
    how="inner"
)

print("학습 웨이퍼·단계 조합:", len(training_keys))
print("검증 웨이퍼·단계 조합:", len(validation_keys))
print("중복된 웨이퍼·단계 조합:", len(overlap_keys))

display(overlap_keys.head(10))

In [ ]:
# 1. 학습 데이터에서 Stage·Chamber별 평균 제거율 계산
group_baseline = (
    model_ready_df
    .groupby(["STAGE", "CHAMBER"])["AVG_REMOVAL_RATE"]
    .mean()
    .reset_index()
    .rename(
        columns={
            "AVG_REMOVAL_RATE": "BASELINE_PREDICTION"
        }
    )
)

# 2. 검증 결과에 기준모델 예측값 연결
external_compare_df = validation_result_df.merge(
    group_baseline,
    on=["STAGE", "CHAMBER"],
    how="left"
)

# 3. 기준모델 외부 성능 계산
baseline_mae = mean_absolute_error(
    external_compare_df["AVG_REMOVAL_RATE"],
    external_compare_df["BASELINE_PREDICTION"]
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        external_compare_df["AVG_REMOVAL_RATE"],
        external_compare_df["BASELINE_PREDICTION"]
    )
)

baseline_r2 = r2_score(
    external_compare_df["AVG_REMOVAL_RATE"],
    external_compare_df["BASELINE_PREDICTION"]
)

# 4. Random Forest와 비교
print("외부 검증 성능 비교")
print()
print("Stage·Chamber 평균 기준모델")
print(f"MAE  : {baseline_mae:.3f}")
print(f"RMSE : {baseline_rmse:.3f}")
print(f"R²   : {baseline_r2:.3f}")
print()
print("Random Forest")
print(f"MAE  : {validation_mae:.3f}")
print(f"RMSE : {validation_rmse:.3f}")
print(f"R²   : {validation_r2:.3f}")
print()
print(
    "MAE 개선율:",
    f"{(baseline_mae - validation_mae) / baseline_mae * 100:.1f}%"
)

In [ ]:
# 1. 학습 데이터에서 Stage·Chamber별 평균 제거율 계산
group_baseline = (
    model_ready_df
    .groupby(["STAGE", "CHAMBER"])["AVG_REMOVAL_RATE"]
    .mean()
    .reset_index()
    .rename(
        columns={
            "AVG_REMOVAL_RATE": "BASELINE_PREDICTION"
        }
    )
)

# 2. 검증 결과에 기준모델 예측값 연결
external_compare_df = validation_result_df.merge(
    group_baseline,
    on=["STAGE", "CHAMBER"],
    how="left"
)

# 3. 기준모델 외부 성능 계산
baseline_mae = mean_absolute_error(
    external_compare_df["AVG_REMOVAL_RATE"],
    external_compare_df["BASELINE_PREDICTION"]
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        external_compare_df["AVG_REMOVAL_RATE"],
        external_compare_df["BASELINE_PREDICTION"]
    )
)

baseline_r2 = r2_score(
    external_compare_df["AVG_REMOVAL_RATE"],
    external_compare_df["BASELINE_PREDICTION"]
)

# 4. Random Forest와 비교
print("외부 검증 성능 비교")
print()
print("Stage·Chamber 평균 기준모델")
print(f"MAE  : {baseline_mae:.3f}")
print(f"RMSE : {baseline_rmse:.3f}")
print(f"R²   : {baseline_r2:.3f}")
print()
print("Random Forest")
print(f"MAE  : {validation_mae:.3f}")
print(f"RMSE : {validation_rmse:.3f}")
print(f"R²   : {validation_r2:.3f}")
print()
print(
    "MAE 개선율:",
    f"{(baseline_mae - validation_mae) / baseline_mae * 100:.1f}%"
)

In [ ]:
group_results = []

for (stage, chamber), group in external_compare_df.groupby(
    ["STAGE", "CHAMBER"]
):
    actual = group["AVG_REMOVAL_RATE"]
    rf_prediction = group["PREDICTED_REMOVAL_RATE"]
    baseline_prediction = group["BASELINE_PREDICTION"]

    group_results.append({
        "STAGE": stage,
        "CHAMBER": chamber,
        "검증_웨이퍼": len(group),

        "기준_MAE": mean_absolute_error(
            actual,
            baseline_prediction
        ),

        "RF_MAE": mean_absolute_error(
            actual,
            rf_prediction
        ),

        "RF_RMSE": np.sqrt(
            mean_squared_error(actual, rf_prediction)
        ),

        "RF_R2": r2_score(
            actual,
            rf_prediction
        )
    })

group_result_df = pd.DataFrame(group_results)

group_result_df["MAE_개선율_%"] = (
    (group_result_df["기준_MAE"] - group_result_df["RF_MAE"])
    / group_result_df["기준_MAE"]
    * 100
)

display(group_result_df.round(3))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 7))

sns.scatterplot(
    data=validation_result_df,
    x="AVG_REMOVAL_RATE",
    y="PREDICTED_REMOVAL_RATE",
    hue="STAGE",
    style="CHAMBER",
    s=65,
    alpha=0.75
)

minimum = min(
    validation_result_df["AVG_REMOVAL_RATE"].min(),
    validation_result_df["PREDICTED_REMOVAL_RATE"].min()
)

maximum = max(
    validation_result_df["AVG_REMOVAL_RATE"].max(),
    validation_result_df["PREDICTED_REMOVAL_RATE"].max()
)

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    "r--",
    label="Perfect Prediction"
)

plt.xlabel("Actual Removal Rate")
plt.ylabel("Predicted Removal Rate")
plt.title(
    "External Validation: Actual vs Predicted\n"
    "MAE=2.435, RMSE=3.268, R²=0.987"
)

plt.legend()
plt.tight_layout()

plt.savefig(
    "../results/external_validation.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
validation_result_df.to_csv(
    "../results/external_validation_results.csv",
    index=False,
    encoding="utf-8-sig"
)

group_result_df.to_csv(
    "../results/external_group_performance.csv",
    index=False,
    encoding="utf-8-sig"
)

print("외부 검증 결과 저장 완료")

In [ ]:
# 절대오차가 큰 웨이퍼 순서로 정렬
external_error_top15 = (
    validation_result_df
    .sort_values("ABS_ERROR", ascending=False)
    .head(15)
)

display(
    external_error_top15[
        [
            "WAFER_ID",
            "STAGE",
            "CHAMBER",
            "AVG_REMOVAL_RATE",
            "PREDICTED_REMOVAL_RATE",
            "ERROR",
            "ABS_ERROR"
        ]
    ]
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(9, 6))

sns.boxplot(
    data=validation_result_df,
    x="STAGE",
    y="ABS_ERROR",
    hue="CHAMBER"
)

plt.axhline(
    validation_mae,
    color="red",
    linestyle="--",
    label=f"Overall MAE = {validation_mae:.3f}"
)

plt.title("External Validation Absolute Error by Process Group")
plt.xlabel("CMP Stage")
plt.ylabel("Absolute Prediction Error")
plt.legend()
plt.tight_layout()

plt.savefig(
    "../results/external_error_by_group.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
group_error_summary = (
    validation_result_df
    .groupby(["STAGE", "CHAMBER"])
    .agg(
        WAFERS=("WAFER_ID", "count"),
        MEAN_ERROR=("ERROR", "mean"),
        MAE=("ABS_ERROR", "mean"),
        MAX_ERROR=("ABS_ERROR", "max")
    )
    .reset_index()
)

display(group_error_summary.round(3))

In [ ]:
import joblib
from pathlib import Path

# 모델 저장 폴더 생성
model_folder = Path("../results/models")
model_folder.mkdir(parents=True, exist_ok=True)

# 1. 학습된 Random Forest 전체 파이프라인 저장
joblib.dump(
    final_model,
    model_folder / "cmp_removal_rate_model.joblib"
)

# 2. 모델 입력 열 이름과 순서 저장
joblib.dump(
    list(X_final.columns),
    model_folder / "cmp_feature_columns.joblib"
)

print("모델 저장 완료")
print("모델:", model_folder / "cmp_removal_rate_model.joblib")
print("입력 열:", model_folder / "cmp_feature_columns.joblib")

In [ ]:
print("모델 파일 존재:", (
    model_folder / "cmp_removal_rate_model.joblib"
).exists())

print("입력 열 파일 존재:", (
    model_folder / "cmp_feature_columns.joblib"
).exists())

In [ ]:
from pathlib import Path
import pandas as pd
import joblib


def predict_new_raw_folder(raw_folder, output_path):
    """
    신규 CMP Raw CSV 폴더를 입력받아
    웨이퍼별 제거율을 예측하고 CSV로 저장하는 함수
    """

    raw_folder = Path(raw_folder)
    output_path = Path(output_path)

    # 1. 저장된 모델과 입력 열 불러오기
    loaded_model = joblib.load(
        "../results/models/cmp_removal_rate_model.joblib"
    )

    feature_columns = joblib.load(
        "../results/models/cmp_feature_columns.joblib"
    )

    # 2. 신규 Raw CSV 파일 찾기
    raw_files = sorted(raw_folder.glob("*.csv"))

    print("발견한 Raw 파일:", len(raw_files))

    if len(raw_files) == 0:
        raise ValueError("지정한 폴더에서 CSV 파일을 찾지 못했습니다.")

    # 3. Raw 파일 결합
    raw_parts = []

    for file in raw_files:
        part = pd.read_csv(file, usecols=all_columns)

        if not part.empty:
            raw_parts.append(part)

    if len(raw_parts) == 0:
        raise ValueError("CSV 파일에 분석할 데이터가 없습니다.")

    new_raw_df = pd.concat(
        raw_parts,
        ignore_index=True
    )

    print("결합된 Raw Data:", new_raw_df.shape)

    # 4. 공정변수 평균 계산
    process_mean = (
        new_raw_df
        .groupby(["WAFER_ID", "STAGE"])[process_columns]
        .mean()
        .add_suffix("_mean")
    )

    # 5. 공정변수 표준편차 계산
    process_std = (
        new_raw_df
        .groupby(["WAFER_ID", "STAGE"])[process_columns]
        .std()
        .add_suffix("_std")
    )

    # 6. 소모품 사용량 평균 계산
    wear_mean = (
        new_raw_df
        .groupby(["WAFER_ID", "STAGE"])[wear_columns]
        .mean()
        .add_suffix("_mean")
    )

    # 7. 챔버 정보 추출
    chamber = (
        new_raw_df
        .groupby(["WAFER_ID", "STAGE"])["CHAMBER"]
        .first()
    )

    # 8. 웨이퍼별 특징표 생성
    new_features = pd.concat(
        [
            chamber,
            process_mean,
            process_std,
            wear_mean
        ],
        axis=1
    ).reset_index()

    print("예측할 웨이퍼:", len(new_features))

    # 9. 모델 입력 열과 순서 맞추기
    X_new = new_features.reindex(
        columns=feature_columns
    ).copy()

    # STAGE는 문자로 변환
    X_new["STAGE"] = X_new["STAGE"].astype(str)

    # 나머지 열은 숫자로 변환
    for column in X_new.columns:
        if column != "STAGE":
            X_new[column] = pd.to_numeric(
                X_new[column],
                errors="coerce"
            ).astype("float64")

    # 10. 제거율 예측
    predictions = loaded_model.predict(X_new)

    # 11. 예측 결과표 생성
    prediction_result = new_features.loc[
        :, ["WAFER_ID", "STAGE", "CHAMBER"]
    ].copy()

    prediction_result["PREDICTED_REMOVAL_RATE"] = predictions

    # 12. 결과 CSV 저장
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    prediction_result.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("예측 완료:", output_path)

    return prediction_result

In [ ]:
automatic_prediction = predict_new_raw_folder(
    raw_folder="../data/PHM2016_CMP_VALIDATION/validation",
    output_path="../results/automatic_validation_prediction.csv"
)

display(automatic_prediction.head(10))

In [ ]:
# 자동화 예측값과 기존 예측값 연결
automation_check_df = automatic_prediction.merge(
    validation_prediction_df[
        [
            "WAFER_ID",
            "STAGE",
            "PREDICTED_REMOVAL_RATE"
        ]
    ],
    on=["WAFER_ID", "STAGE"],
    how="inner",
    suffixes=("_AUTO", "_MANUAL")
)

# 두 예측값의 차이 계산
automation_check_df["PREDICTION_DIFFERENCE"] = (
    automation_check_df["PREDICTED_REMOVAL_RATE_AUTO"]
    - automation_check_df["PREDICTED_REMOVAL_RATE_MANUAL"]
).abs()

print("자동 예측 웨이퍼:", len(automatic_prediction))
print("비교된 웨이퍼:", len(automation_check_df))
print(
    "예측값 최대 차이:",
    automation_check_df["PREDICTION_DIFFERENCE"].max()
)

display(automation_check_df.head(10))

In [ ]:
automatic_output_path = Path(
    "../results/automatic_validation_prediction.csv"
)

print("결과 파일 존재:", automatic_output_path.exists())

saved_prediction_df = pd.read_csv(
    automatic_output_path
)

print("저장된 결과 크기:", saved_prediction_df.shape)
display(saved_prediction_df.head())

In [ ]:
# 터미널 프로그램이 생성한 결과 불러오기
terminal_prediction_df = pd.read_csv(
    "../results/terminal_prediction.csv"
)

# 기존 수동 예측 결과와 연결
terminal_check_df = terminal_prediction_df.merge(
    validation_prediction_df[
        [
            "WAFER_ID",
            "STAGE",
            "PREDICTED_REMOVAL_RATE"
        ]
    ],
    on=["WAFER_ID", "STAGE"],
    how="inner",
    suffixes=("_TERMINAL", "_MANUAL")
)

# 예측값 차이 계산
terminal_check_df["DIFFERENCE"] = (
    terminal_check_df["PREDICTED_REMOVAL_RATE_TERMINAL"]
    - terminal_check_df["PREDICTED_REMOVAL_RATE_MANUAL"]
).abs()

print("터미널 예측 웨이퍼:", len(terminal_prediction_df))
print("비교된 웨이퍼:", len(terminal_check_df))
print("예측값 최대 차이:", terminal_check_df["DIFFERENCE"].max())

In [ ]:
# 분석할 공정 그룹 선택
split_group_df = model_ready_df[
    (model_ready_df["STAGE"].astype(str) == "A")
    & (model_ready_df["CHAMBER"] == 4.0)
].copy()

# Split 후보 공정변수 3개
split_variables = [
    "MAIN_OUTER_AIR_BAG_PRESSURE_mean",
    "SLURRY_FLOW_LINE_C_mean",
    "HEAD_ROTATION_mean"
]

# 실제 데이터의 변수 분포 확인
split_range_df = (
    split_group_df[split_variables]
    .quantile([0.10, 0.25, 0.50, 0.75, 0.90])
    .T
)

split_range_df.columns = [
    "10%",
    "25%",
    "50%(중앙값)",
    "75%",
    "90%"
]

print("Stage A / Chamber 4 웨이퍼:", len(split_group_df))
display(split_range_df.round(3))

In [ ]:
# Stage A / Chamber 4의 대표적인 조건 생성
base_condition = (
    split_group_df[X_final.columns]
    .copy()
)

# STAGE는 최빈값, 나머지 변수는 중앙값 사용
representative_condition = {}

for column in X_final.columns:
    if column == "STAGE":
        representative_condition[column] = "A"
    elif column == "CHAMBER":
        representative_condition[column] = 4.0
    else:
        representative_condition[column] = pd.to_numeric(
            base_condition[column],
            errors="coerce"
        ).median()

# Split 조건표 생성
split_cases = []

for variable in split_variables:

    low_value = split_group_df[variable].quantile(0.10)
    mid_value = split_group_df[variable].quantile(0.50)
    high_value = split_group_df[variable].quantile(0.90)

    for level, value in [
        ("Low_10%", low_value),
        ("Mid_50%", mid_value),
        ("High_90%", high_value)
    ]:
        condition = representative_condition.copy()

        # 선택한 변수 하나만 변경
        condition[variable] = value

        condition["SPLIT_VARIABLE"] = variable
        condition["SPLIT_LEVEL"] = level
        condition["SPLIT_VALUE"] = value

        split_cases.append(condition)

split_simulation_df = pd.DataFrame(split_cases)

# 모델 입력용 표
X_split = split_simulation_df[
    X_final.columns
].copy()

X_split["STAGE"] = X_split["STAGE"].astype(str)

for column in X_split.columns:
    if column != "STAGE":
        X_split[column] = pd.to_numeric(
            X_split[column],
            errors="coerce"
        ).astype("float64")

# Split 조건별 제거율 예측
split_simulation_df["PREDICTED_REMOVAL_RATE"] = (
    final_model.predict(X_split)
)

# 보기 쉬운 결과표
split_result_df = split_simulation_df[
    [
        "SPLIT_VARIABLE",
        "SPLIT_LEVEL",
        "SPLIT_VALUE",
        "PREDICTED_REMOVAL_RATE"
    ]
].copy()

display(split_result_df.round(3))

In [ ]:
# Pressure와 Slurry C의 Low / Mid / High 값
pressure_levels = {
    "Low": split_group_df[
        "MAIN_OUTER_AIR_BAG_PRESSURE_mean"
    ].quantile(0.10),

    "Mid": split_group_df[
        "MAIN_OUTER_AIR_BAG_PRESSURE_mean"
    ].quantile(0.50),

    "High": split_group_df[
        "MAIN_OUTER_AIR_BAG_PRESSURE_mean"
    ].quantile(0.90)
}

slurry_levels = {
    "Low": split_group_df[
        "SLURRY_FLOW_LINE_C_mean"
    ].quantile(0.10),

    "Mid": split_group_df[
        "SLURRY_FLOW_LINE_C_mean"
    ].quantile(0.50),

    "High": split_group_df[
        "SLURRY_FLOW_LINE_C_mean"
    ].quantile(0.90)
}

# 3 × 3 = 총 9개 가상 조건 생성
two_factor_cases = []

for pressure_level, pressure_value in pressure_levels.items():

    for slurry_level, slurry_value in slurry_levels.items():

        condition = representative_condition.copy()

        condition[
            "MAIN_OUTER_AIR_BAG_PRESSURE_mean"
        ] = pressure_value

        condition[
            "SLURRY_FLOW_LINE_C_mean"
        ] = slurry_value

        condition["PRESSURE_LEVEL"] = pressure_level
        condition["SLURRY_LEVEL"] = slurry_level
        condition["PRESSURE_VALUE"] = pressure_value
        condition["SLURRY_VALUE"] = slurry_value

        two_factor_cases.append(condition)

two_factor_df = pd.DataFrame(two_factor_cases)

# 모델 입력값 준비
X_two_factor = two_factor_df[
    X_final.columns
].copy()

X_two_factor["STAGE"] = (
    X_two_factor["STAGE"].astype(str)
)

for column in X_two_factor.columns:
    if column != "STAGE":
        X_two_factor[column] = pd.to_numeric(
            X_two_factor[column],
            errors="coerce"
        ).astype("float64")

# 9개 조건의 제거율 예측
two_factor_df["PREDICTED_REMOVAL_RATE"] = (
    final_model.predict(X_two_factor)
)

# 결과표
two_factor_result = two_factor_df[
    [
        "PRESSURE_LEVEL",
        "PRESSURE_VALUE",
        "SLURRY_LEVEL",
        "SLURRY_VALUE",
        "PREDICTED_REMOVAL_RATE"
    ]
].copy()

display(two_factor_result.round(3))

In [ ]:
split_window_table = two_factor_result.pivot(
    index="PRESSURE_LEVEL",
    columns="SLURRY_LEVEL",
    values="PREDICTED_REMOVAL_RATE"
)

# Low → Mid → High 순서 정렬
split_window_table = split_window_table.reindex(
    index=["Low", "Mid", "High"],
    columns=["Low", "Mid", "High"]
)

display(split_window_table.round(3))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7, 5))

sns.heatmap(
    split_window_table,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd"
)

plt.title(
    "Model-based Split Simulation\n"
    "Stage A / Chamber 4"
)

plt.xlabel("Slurry Flow C Level")
plt.ylabel("Outer Air Bag Pressure Level")

plt.tight_layout()
plt.show()

In [ ]:
# 제거율이 높은 순서로 정렬
ranked_split_df = (
    two_factor_result
    .sort_values(
        "PREDICTED_REMOVAL_RATE",
        ascending=False
    )
    .reset_index(drop=True)
)

ranked_split_df["순위"] = (
    ranked_split_df.index + 1
)

display(
    ranked_split_df[
        [
            "순위",
            "PRESSURE_LEVEL",
            "PRESSURE_VALUE",
            "SLURRY_LEVEL",
            "SLURRY_VALUE",
            "PREDICTED_REMOVAL_RATE"
        ]
    ].round(3)
)

# 9개 조건의 예측 제거율 차이
maximum_rate = ranked_split_df[
    "PREDICTED_REMOVAL_RATE"
].max()

minimum_rate = ranked_split_df[
    "PREDICTED_REMOVAL_RATE"
].min()

print(f"최고 예상 제거율: {maximum_rate:.3f}")
print(f"최저 예상 제거율: {minimum_rate:.3f}")
print(f"Split 조건 간 최대 차이: {maximum_rate - minimum_rate:.3f}")

In [ ]:
plt.figure(figsize=(8, 5))

sns.lineplot(
    data=two_factor_result,
    x="PRESSURE_LEVEL",
    y="PREDICTED_REMOVAL_RATE",
    hue="SLURRY_LEVEL",
    marker="o",
    sort=False
)

plt.title(
    "Pressure × Slurry Interaction Simulation\n"
    "Stage A / Chamber 4"
)

plt.xlabel("Outer Air Bag Pressure Level")
plt.ylabel("Predicted Removal Rate")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

pressure_column = "MAIN_OUTER_AIR_BAG_PRESSURE_mean"
slurry_column = "SLURRY_FLOW_LINE_C_mean"

# 실제 Stage A / Chamber 4의 압력·슬러리 조합
real_conditions = (
    split_group_df[
        [pressure_column, slurry_column]
    ]
    .apply(pd.to_numeric, errors="coerce")
    .dropna()
)

# 9개의 가상 Split 조합
simulated_conditions = (
    two_factor_df[
        [pressure_column, slurry_column]
    ]
    .apply(pd.to_numeric, errors="coerce")
)

# 압력과 슬러리의 단위 차이를 제거하기 위해 표준화
scaler = StandardScaler()

real_scaled = scaler.fit_transform(real_conditions)
simulated_scaled = scaler.transform(simulated_conditions)

# 각 가상 조건에서 가장 가까운 실제 웨이퍼 조건 찾기
nearest_model = NearestNeighbors(n_neighbors=1)
nearest_model.fit(real_scaled)

nearest_distance, nearest_index = nearest_model.kneighbors(
    simulated_scaled
)

# 결과표에 실제 데이터와의 거리 추가
support_check_df = two_factor_result.copy()

support_check_df["NEAREST_REAL_DISTANCE"] = (
    nearest_distance.flatten()
)

support_check_df["CLOSEST_REAL_PRESSURE"] = (
    real_conditions.iloc[
        nearest_index.flatten()
    ][pressure_column].to_numpy()
)

support_check_df["CLOSEST_REAL_SLURRY"] = (
    real_conditions.iloc[
        nearest_index.flatten()
    ][slurry_column].to_numpy()
)

# 거리가 가까운 순으로 정렬
support_check_df = support_check_df.sort_values(
    "NEAREST_REAL_DISTANCE"
).reset_index(drop=True)

display(support_check_df.round(3))

In [ ]:
distance_q1 = support_check_df[
    "NEAREST_REAL_DISTANCE"
].quantile(0.33)

distance_q2 = support_check_df[
    "NEAREST_REAL_DISTANCE"
].quantile(0.67)


def classify_support(distance):
    if distance <= distance_q1:
        return "높음"
    elif distance <= distance_q2:
        return "보통"
    else:
        return "낮음"


support_check_df["DATA_SUPPORT"] = (
    support_check_df["NEAREST_REAL_DISTANCE"]
    .apply(classify_support)
)

display(
    support_check_df[
        [
            "PRESSURE_LEVEL",
            "SLURRY_LEVEL",
            "PRESSURE_VALUE",
            "SLURRY_VALUE",
            "PREDICTED_REMOVAL_RATE",
            "NEAREST_REAL_DISTANCE",
            "DATA_SUPPORT"
        ]
    ].round(3)
)